# Build the image–text cross-modal retrieval benchmark (step 1)

This notebook builds `benchmark/benchmark.csv`: **one row per photo**, pairing each image in `photos/` with the textual description of the phenological stage encoded in its filename.

## clone the BBCH projet

clone [https://forge.inrae.fr/bsv/phenologicalstages](https://forge.inrae.fr/bsv/phenologicalstages)

then run `copy_photos.sh`

## Filename conventions

Photos are named `<crop>#<rest>.<ext>`. Two naming schemes exist today:

1. **BBCH scheme** (e.g. `citrus#photo_citrus_bbch_59_1.jpg`)
   - `59` = BBCH **secondary** stage code (2 digits)
   - the **principal** stage is the first digit (`5`)
   - trailing `_1` is a photo variant number, not part of the stage code
2. **Terres Inovia (TI) scheme** (e.g. `oilseedFlax#terres_inovia_Zoe_LeBihan_D1_2_photo_oilseedFlax.jpg`, `oilseedRape#terres_inovia_E_photo_oilseed_rape.png`)
   - the stage code is the **last token matching an uppercase letter followed by optional digits** (`D1`, `E5`, `E`, `C2`, …)
   - a purely numeric token right after the code (`_2`) is a photo variant

## Stage descriptions

Stage codes are expanded via the Excel files in `stage_excels/`. Their layout is known from the Cellfie rule files (`<crop>/cellfie/rulesInstanceGeneration_*.json`):

- Language sheets (`FR*`, `EN*`, `ES*`): col A = stage code, B = definition, C = prefLabel, D/E = alternative labels.
- `info` sheet: col A = code, B = precedes, C = follows, D = types, E+ = alignment targets (e.g. BBCH equivalents).

## Output CSV schema

`photo_name, crop_name, bbch_principal_stage_id, bbch_secondary_stage_id, ti_stage_id, principal_desc_fr, principal_desc_en, secondary_desc_fr, secondary_desc_en`

- citrus rows: BBCH ids filled, `ti_stage_id` empty.
- oilseedFlax / oilseedRape rows: `ti_stage_id` filled; BBCH ids derived from the alignment columns when possible (otherwise empty).
- Photos whose stage cannot be resolved are reported in `benchmark/unmatched.csv`.

In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Step 0 — Configuration

Paths and the crop → Excel mapping. Each crop lists which Excel files hold its **principal** and **secondary** stages. For oilseedFlax the TI code may exist in the winter scale, the spring scale, or both; all candidate files are listed and every match is kept (descriptions are concatenated with ` || ` and tagged by scale).

In [2]:
from pathlib import Path
from collections import defaultdict
import os
import pickle
import re
import openpyxl
import pandas as pd

ROOT = CASE_STUDY_ROOT
PHOTOS_DIR = ROOT / "photos"
EXCEL_DIR = ROOT / "stage_excels"
OUT_DIR = ROOT / "benchmark"
OUT_DIR.mkdir(exist_ok=True)

# Intermediate artifacts are cached here, same convention as notebook 29 (repo-root vars/).
VARS_DIR = CASE_STUDY_ROOT / "vars"
VARS_DIR.mkdir(exist_ok=True)

# Crop candidate list for the image -> crop retrieval task (task 1 of notebook 32).
# Lives next to this notebook: 5.BBCH case study/cropnames.txt
CROP_NAMES_PATH = ROOT / "cropnames.txt"
assert CROP_NAMES_PATH.exists(), f"cropnames.txt not found at {CROP_NAMES_PATH}"


def pkl_vars(varname, filename):
    """Pickle a variable into VARS_DIR (same helper as notebook 29)."""
    with open(VARS_DIR / filename, "wb") as file:
        pickle.dump(varname, file)


def reload_vars(filename):
    """Load a variable previously written by pkl_vars()."""
    with open(VARS_DIR / filename, "rb") as file:
        return pickle.load(file)


# Best effort: derive BBCH ids for TI-coded photos from the alignment columns
# of the Excel `info` sheets (skos mapping targets). Set to False to leave empty.
DERIVE_BBCH_FROM_ALIGNMENTS = True


CROP_CONFIG = {
    "citrus": {
        "kind": "bbch",
        "principal": ["citrus_bbch_principalStages.xlsx"],
        "secondary": ["citrus_bbch_secondaryStages.xlsx"],
    },
    "oilseedFlax": {
        "kind": "ti",
        "principal": ["oilseedFlax_SmithFroment_principalStages.xlsx"],
        "secondary": [
            "winterOilseedFlax_ti_secondaryStages.xlsx",
            "springOilseedFlax_ti_secondaryStages.xlsx",
            "oilseedFlax_SmithFroment_secondaryStages.xlsx",
        ],
    },
    "oilseedRape": {
        "kind": "ti",
        "principal": [],
        "secondary": ["oilseed_rape_ti_secondaryStages.xlsx"],
    },
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
print(f"Photos found: {len([p for p in PHOTOS_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS])}")
print(f"VARS_DIR: {VARS_DIR}")
print(f"CROP_NAMES_PATH: {CROP_NAMES_PATH}")

Photos found: 97
VARS_DIR: /projects/agrifutur/sji/bsv-crossmodalIR/vars
CROP_NAMES_PATH: /projects/agrifutur/sji/bsv-crossmodalIR/bbch-benchmark/cropnames.txt


## Step 1 — Parse the photo filenames

Extract from each filename: the **crop** (prefix before `#`), the **stage code**, the **scheme kind** (`bbch` or `ti`) and the **photo variant**. BBCH filenames are matched with a dedicated regex; for TI filenames the stage code is the last `_`-separated token of the form `^[A-Z]\d*$` (person names like `Zoe`/`LeBihan` contain lowercase letters and therefore never match; a purely numeric token following the code is the variant).

In [3]:
BBCH_RE = re.compile(r"^photo_.*_bbch_(?P<code>\d{2})_(?P<variant>\d+)$")
TI_CODE_RE = re.compile(r"^[A-Z]\d*$")


def parse_filename(filename: str):
    """Return {crop, kind, stage_code, variant, scale} or None if unrecognized."""
    stem = Path(filename).stem
    if "#" not in stem:
        return None
    crop, rest = stem.split("#", 1)

    m = BBCH_RE.match(rest)
    if m:
        return {
            "crop": crop,
            "kind": "bbch",
            "stage_code": m.group("code"),
            "variant": m.group("variant"),
            "scale": "bbch",
        }

    tokens = rest.split("_")
    code, variant = None, ""
    for i, tok in enumerate(tokens):
        if TI_CODE_RE.match(tok):
            code = tok
            variant_tokens = []
            for t in tokens[i + 1:]:          # numeric tokens right after the code
                if t.isdigit():
                    variant_tokens.append(t)
                else:
                    break
            variant = "_".join(variant_tokens)
    if code:
        return {
            "crop": crop,
            "kind": "ti",
            "stage_code": code,
            "variant": variant,
            "scale": "terres_inovia",
        }
    return None


parsed = []
unparsed = []
for photo in sorted(PHOTOS_DIR.iterdir()):
    if photo.suffix.lower() not in IMAGE_EXTS:
        continue
    info = parse_filename(photo.name)
    if info is None:
        unparsed.append(photo.name)
    else:
        parsed.append({"photo_name": photo.name, **info})

df_parsed = pd.DataFrame(parsed)
print(f"Parsed: {len(df_parsed)}   Unparsed: {len(unparsed)}")
if unparsed:
    print("Unrecognized filenames:", unparsed)
df_parsed.groupby(["crop", "kind", "stage_code"]).size().reset_index(name="n_photos")

Parsed: 97   Unparsed: 0


,crop,kind,stage_code,n_photos
0,citrus,bbch,00,1
1,citrus,bbch,01,1
2,citrus,bbch,03,1
3,citrus,bbch,07,2
4,citrus,bbch,09,4
5,citrus,bbch,10,3
6,citrus,bbch,15,2
7,citrus,bbch,19,1
8,citrus,bbch,31,2
9,citrus,bbch,32,2


## Step 2 — Load the stage descriptions from the Excel files

For every Excel file referenced in the configuration, read all sheets and classify them by name:

- `INFO` → structural sheet (code, precedes, follows, types, alignments in columns E+).
- `FR*` / `EN*`  → language sheets (code, definition, prefLabel, altLabel, altLabel).

The header row is auto-detected by scanning the first rows for a cell in column A that looks like a stage code (1–2 digits, or an uppercase letter followed by digits). The result is a dict `code -> {"fr": {...}, "en": {...}, "info": {...}}` per Excel file.

In [4]:
LANG_PREFIXES = {"FR": "fr", "EN": "en"}
CODE_RE = re.compile(r"^(\d{1,2}|[A-Z]\d*)$")


def classify_sheet(name: str):
    n = name.strip().upper()
    if n == "INFO":
        return "info"
    for prefix, lang in LANG_PREFIXES.items():
        if n.startswith(prefix):
            return lang
    return None


def _find_header_row(ws, max_scan: int = 6):
    """First row whose column A value looks like a stage code."""
    for r, row in enumerate(
        ws.iter_rows(min_row=1, max_row=max_scan, max_col=1, values_only=True),
        start=1,
    ):
        v = row[0]
        if v is not None and CODE_RE.match(str(v).strip()):
            return r
    return None


def load_stage_excel(path: Path):
    """Load one stage Excel file -> {code: {lang: {prefLabel, definition, altLabels}, info: {...}}}."""
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    stages: dict = defaultdict(dict)
    for ws in wb.worksheets:
        role = classify_sheet(ws.title)
        if role is None:
            continue
        header = _find_header_row(ws)
        if header is None:
            continue
        for row in ws.iter_rows(min_row=header, values_only=True):
            if row[0] is None:
                continue
            code = str(row[0]).strip()
            if not CODE_RE.match(code):
                continue
            if role == "info":
                stages[code]["info"] = {
                    "precedes": row[1],
                    "follows": row[2],
                    "types": row[3],
                    "alignments": [c for c in row[4:] if c not in (None, "")],
                }
            else:
                stages[code][role] = {
                    "definition": row[1],
                    "prefLabel": row[2],
                    "altLabels": [c for c in row[3:5] if c not in (None, "")],
                }
    wb.close()
    return dict(stages)


# Load every Excel file used by the configuration
excel_files = sorted(
    {f for cfg in CROP_CONFIG.values() for f in cfg["principal"] + cfg["secondary"]}
)
loaded = {}
for fname in excel_files:
    path = EXCEL_DIR / fname
    if not path.exists():
        print(f"WARNING: missing Excel file {path}")
        continue
    loaded[fname] = load_stage_excel(path)
    print(f"{fname}: {len(loaded[fname])} stages loaded")

citrus_bbch_principalStages.xlsx: 8 stages loaded
citrus_bbch_secondaryStages.xlsx: 35 stages loaded
oilseedFlax_SmithFroment_principalStages.xlsx: 9 stages loaded
oilseedFlax_SmithFroment_secondaryStages.xlsx: 54 stages loaded
oilseed_rape_ti_secondaryStages.xlsx: 28 stages loaded
springOilseedFlax_ti_secondaryStages.xlsx: 26 stages loaded
winterOilseedFlax_ti_secondaryStages.xlsx: 27 stages loaded


## Step 3 — Build the benchmark rows

One row per photo:

- **BBCH photos** (citrus): secondary id = 2-digit code, principal id = first digit; descriptions looked up in the corresponding principal/secondary Excel files.
- **TI photos** (oilseedFlax, oilseedRape): `ti_stage_id` = extracted code; the code is searched in every secondary Excel of the crop (a flax code may match both the winter and spring scales — all matches are kept, descriptions joined with ` || `). BBCH ids are optionally derived from the alignment columns of the `info` sheets (values ending in `BBCH<2 digits>`); descriptions for derived BBCH ids are filled only when the id exists in the crop's principal Excel, otherwise left empty.
- Unresolvable photos are collected with a reason and excluded from the CSV.

In [5]:
BBCH_ALIGN_RE = re.compile(r"BBCH(\d{1,2})$", re.IGNORECASE)

COLUMNS = [
    "photo_name", "crop_name",
    "bbch_principal_stage_id", "bbch_secondary_stage_id", "ti_stage_id",
    "principal_desc_fr", "principal_desc_en",
    "secondary_desc_fr", "secondary_desc_en",
]


def make_description(stage: dict, lang: str) -> str:
    """prefLabel + definition for one language, '' when unavailable."""
    s = (stage or {}).get(lang)
    if not s:
        return ""
    pref = str(s.get("prefLabel") or "").strip()
    defn = str(s.get("definition") or "").strip()
    if pref and defn:
        return f"{pref}. {defn}"
    return pref or defn


def bbch_from_alignments(stage: dict) -> str:
    """First BBCH 2-digit code found in the alignment targets of the info sheet."""
    info = (stage or {}).get("info") or {}
    for target in info.get("alignments", []):
        m = BBCH_ALIGN_RE.search(str(target))
        if m:
            return m.group(1).zfill(2)
    return ""


def empty_row(photo_name: str, crop: str) -> dict:
    row = {c: "" for c in COLUMNS}
    row["photo_name"], row["crop_name"] = photo_name, crop
    return row


rows, unmatched = [], []
for item in parsed:
    crop, code = item["crop"], item["stage_code"]
    cfg = CROP_CONFIG.get(crop)
    if cfg is None:
        unmatched.append({"photo_name": item["photo_name"], "reason": f"no config for crop '{crop}'"})
        continue
    row = empty_row(item["photo_name"], crop)

    if item["kind"] == "bbch":
        pri_code, sec_code = code[0], code
        row["bbch_principal_stage_id"] = pri_code
        row["bbch_secondary_stage_id"] = sec_code

        pri_stage = None
        for fname in cfg["principal"]:
            pri_stage = loaded.get(fname, {}).get(pri_code)
            if pri_stage:
                break
        sec_stage = None
        for fname in cfg["secondary"]:
            sec_stage = loaded.get(fname, {}).get(sec_code)
            if sec_stage:
                break
        if pri_stage is None or sec_stage is None:
            missing = "principal" if pri_stage is None else "secondary"
            unmatched.append({"photo_name": item["photo_name"],
                              "reason": f"{missing} stage '{code}' not found in Excel"})
            continue
        for lang in ("fr", "en"):
            row[f"principal_desc_{lang}"] = make_description(pri_stage, lang)
            row[f"secondary_desc_{lang}"] = make_description(sec_stage, lang)

    else:  # ti
        row["ti_stage_id"] = code
        matches = []
        for fname in cfg["secondary"]:
            stage = loaded.get(fname, {}).get(code)
            if stage:
                matches.append((fname, stage))
        if not matches:
            unmatched.append({"photo_name": item["photo_name"],
                              "reason": f"TI stage '{code}' not found in any secondary Excel"})
            continue
        for lang in ("fr", "en"):
            descs = [make_description(stage, lang) for _, stage in matches]
            row[f"secondary_desc_{lang}"] = " || ".join(d for d in descs if d)

        if DERIVE_BBCH_FROM_ALIGNMENTS:
            bbch_codes = sorted({bbch_from_alignments(stage) for _, stage in matches} - {""})
            if bbch_codes:
                sec = bbch_codes[0]
                row["bbch_secondary_stage_id"] = sec
                row["bbch_principal_stage_id"] = sec[0]
                # fill BBCH descriptions only if the id exists in the crop's principal Excel
                for fname in cfg["principal"]:
                    pri_stage = loaded.get(fname, {}).get(sec[0])
                    if pri_stage:
                        for lang in ("fr", "en"):
                            row[f"principal_desc_{lang}"] = make_description(pri_stage, lang)
                        break
    rows.append(row)

df_bench = pd.DataFrame(rows, columns=COLUMNS)

# Remove "(source BBCH monograph)" from the descriptions, if present

SOURCE_TAG_RE = re.compile(
    r"\s*\([^()]*?(?:source|traduit\s+de|Smith\s*&\s*Froment)[^()]*?\)",
    re.IGNORECASE,
)

DESC_COLS = [
    "principal_desc_fr", "principal_desc_en",
    "secondary_desc_fr", "secondary_desc_en",
]

for col in DESC_COLS:
    df_bench[col] = df_bench[col].str.replace(SOURCE_TAG_RE, "", regex=True).str.strip()

print(f"Benchmark rows: {len(df_bench)}   Unmatched: {len(unmatched)}")
df_bench

Benchmark rows: 94   Unmatched: 3


,photo_name,crop_name,bbch_principal_stage_id,bbch_secondary_stage_id,ti_stage_id,principal_desc_fr,principal_desc_en,secondary_desc_fr,secondary_desc_en
0,citrus#photo_citrus_bbch_00_1.jpg,citrus,0,00,,stade principal BBCH 0 des agrumes. Développem...,principal stage BBCH 0 of citrus. Sprouting/Bu...,stade secondaire BBCH 00 des agrumes. Dormance...,secondary stage BBCH 00 of citrus. Dormancy: l...
1,citrus#photo_citrus_bbch_01_1.jpg,citrus,0,01,,stade principal BBCH 0 des agrumes. Développem...,principal stage BBCH 0 of citrus. Sprouting/Bu...,stade secondaire BBCH 01 des agrumes. Début du...,secondary stage BBCH 01 of citrus. Beginning o...
2,citrus#photo_citrus_bbch_03_1.jpg,citrus,0,03,,stade principal BBCH 0 des agrumes. Développem...,principal stage BBCH 0 of citrus. Sprouting/Bu...,stade secondaire BBCH 03 des agrumes. Fin du g...,secondary stage BBCH 03 of citrus. End of bud ...
3,citrus#photo_citrus_bbch_07_1.jpg,citrus,0,07,,stade principal BBCH 0 des agrumes. Développem...,principal stage BBCH 0 of citrus. Sprouting/Bu...,stade secondaire BBCH 07 des agrumes. Début de...,secondary stage BBCH 07 of citrus. Beginnin of...
4,citrus#photo_citrus_bbch_07_2.jpeg,citrus,0,07,,stade principal BBCH 0 des agrumes. Développem...,principal stage BBCH 0 of citrus. Sprouting/Bu...,stade secondaire BBCH 07 des agrumes. Début de...,secondary stage BBCH 07 of citrus. Beginnin of...
...,...,...,...,...,...,...,...,...,...
89,oilseedFlax#terres_inovia_Zoe_LeBihan_F1_photo...,oilseedFlax,6,60,F1,stade principal BBCH 6 du lin. Floraison et fo...,principal stage BBCH 6 of linen. Flowering and...,stade secondaire Terres Inovia F1 du lin. Débu...,Terres Inovia secondary stage F1 of oilseed fl...
90,oilseedFlax#terres_inovia_Zoe_LeBihan_F5_photo...,oilseedFlax,6,62,F5,stade principal BBCH 6 du lin. Floraison et fo...,principal stage BBCH 6 of linen. Flowering and...,stade secondaire Terres Inovia F5 du lin. Plei...,Terres Inovia secondary stage F5 of oilseed fl...
91,oilseedFlax#terres_inovia_Zoe_LeBihan_H9_photo...,oilseedFlax,8,89,H9,stade principal BBCH 8 du lin. Maturation de l...,principal stage BBCH 8 of linen. Capsule and s...,stade secondaire Terres Inovia H9 du lin. Tout...,Terres Inovia secondary stage H9 of oilseed fl...
92,oilseedRape#terres_inovia_C2_photo_oilseed_rap...,oilseedRape,,,C2,,,stade secondaire Terres Inovia C2 du colza. En...,Terres Inovia secondary stage C2 of oilseed ra...


## Step 4 — Per-crop stage lists (candidate pools for the IR benchmark)

The retrieval benchmark in notebook 32 needs, for every crop, the **set of candidate stages** an image can be matched against. Two levels are built independently:

- **principal** — the coarse BBCH principal stage (single digit for BBCH crops, `A`–`H` for Terres Inovia crops)
- **secondary** — the fine BBCH secondary stage (2 digits) or the Terres Inovia code (`D1`, `E5`, …)

Each row carries the stage id plus its French and English descriptions, so the same table serves both the candidate pool and the retrieval text.

The crop-level candidate list (task 1: image → crop) is read from `cropnames.txt` and kept **complete** (all 32 crops): every photo is ranked against the full crop vocabulary, so R@K stays comparable across models.

Outputs (written in step 5): `benchmark/stage_lists.csv` and `vars/bbch_stage_lists.pkl`.

In [6]:
STAGE_LIST_COLUMNS = ["crop_name", "level", "stage_id", "desc_fr", "desc_en"]


def collect_stage_list(crop: str, cfg: dict) -> list:
    """Every stage code present in the crop's Excel files, as principal/secondary rows.

    Principal stages come from the `principal` files, secondary stages from the `secondary`
    files. A code found in several files (e.g. a flax TI code in both the winter and spring
    scale) is emitted once per file, with the descriptions joined by ` || `.
    """
    rows = []
    for level in ("principal", "secondary"):
        per_code = defaultdict(list)
        for fname in cfg[level]:
            for code, stage in loaded.get(fname, {}).items():
                per_code[code].append(stage)
        for code in sorted(per_code):
            stages = per_code[code]
            row = {"crop_name": crop, "level": level, "stage_id": code}
            for lang in ("fr", "en"):
                descs = [make_description(s, lang) for s in stages]
                row[f"desc_{lang}"] = " || ".join(d for d in descs if d)
            rows.append(row)
    return rows


stage_rows = []
for crop, cfg in CROP_CONFIG.items():
    stage_rows.extend(collect_stage_list(crop, cfg))

df_stage_lists = pd.DataFrame(stage_rows, columns=STAGE_LIST_COLUMNS)

# (crop, level) -> list of candidate stage rows; this is the pool notebook 32 ranks against
stage_lists = defaultdict(list)
for row in stage_rows:
    stage_lists[(row["crop_name"], row["level"])].append(row)
stage_lists = dict(stage_lists)

# --- crop-level candidate list (task 1: image -> crop) ---------------------
# The benchmark ranks every photo against the FULL crop vocabulary: all 32 lines of
# cropnames.txt. Never shrink this pool -- a citrus photo must be able to be confused
# with any of the other 31 crops, otherwise R@K is not comparable across models.
crop_candidates = [
    line.strip() for line in CROP_NAMES_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# Diagnostic only: which crops actually have photos on disk.
photo_crops = sorted(
    {p.name.split("#", 1)[0] for p in PHOTOS_DIR.iterdir()
     if p.suffix.lower() in IMAGE_EXTS and "#" in p.name}
)
missing_from_cropnames = [c for c in photo_crops if c not in crop_candidates]

print(f"Crop candidates: {len(crop_candidates)} (all lines of cropnames.txt)")
print(f"Crops with photos on disk: {photo_crops}")
if missing_from_cropnames:
    print(f"WARNING: crops with photos but absent from cropnames.txt: {missing_from_cropnames}")

# Every benchmark photo must be rankable against the whole pool: its own crop has to be in it.
assert set(df_bench["crop_name"]).issubset(set(crop_candidates)), (
    "Some benchmark crops are missing from the candidate pool: "
    f"{sorted(set(df_bench['crop_name']) - set(crop_candidates))}"
)

df_stage_lists

Crop candidates: 32 (all lines of cropnames.txt)
Crops with photos on disk: ['citrus', 'oilseedFlax', 'oilseedRape']


,crop_name,level,stage_id,desc_fr,desc_en
0,citrus,principal,0,stade principal BBCH 0 des agrumes. Développem...,principal stage BBCH 0 of citrus. Sprouting/Bu...
1,citrus,principal,1,stade principal BBCH 1 des agrumes. Développem...,principal stage BBCH 1 of citrus. Leaf develop...
2,citrus,principal,3,stade principal BBCH 3 des agrumes. Développem...,principal stage BBCH 3 of citrus. Shoot develo...
3,citrus,principal,5,stade principal BBCH 5 des agrumes. Développem...,principal stage BBCH 5 of citrus. Inflorescenc...
4,citrus,principal,6,stade principal BBCH 6 des agrumes. La florais...,principal stage BBCH 6 of citrus. Flowering (s...
...,...,...,...,...,...
156,oilseedRape,secondary,G1,stade secondaire Terres Inovia G1 du colza. Ch...,Terres Inovia secondary stage G1 of oilseed ra...
157,oilseedRape,secondary,G2,stade secondaire Terres Inovia G2 du colza. Le...,Terres Inovia secondary stage G2 of oilseed ra...
158,oilseedRape,secondary,G3,stade secondaire Terres Inovia G3 du colza. Le...,Terres Inovia secondary stage G3 of oilseed ra...
159,oilseedRape,secondary,G4,stade secondaire Terres Inovia G4 du colza. Le...,Terres Inovia secondary stage G4 of oilseed ra...


## Step 5 — Clean the descriptions and write all artifacts

`SOURCE_TAG_RE` (defined in step 3) is applied to the candidate pools here, then every artifact is written exactly once:

| artifact | content |
|---|---|
| `benchmark/benchmark.csv` | one row per photo, cleaned descriptions |
| `benchmark/unmatched.csv` | photos whose stage could not be resolved |
| `benchmark/stage_lists.csv` | candidate pools per `(crop, level)` |
| `vars/df_bench.pkl` | the benchmark frame |
| `vars/bbch_stage_lists.pkl` | the stage-list frame |
| `vars/bbch_stage_lists_dict.pkl` | `(crop, level) -> [stage rows]` |
| `vars/bbch_crop_candidates.pkl` | the 32-crop candidate list |

All CSVs are plain UTF-8 with `index=False`.

In [7]:
# SOURCE_TAG_RE is defined once, in step 3, and already applied to df_bench there.
# Here it is applied to the candidate pools, then every artifact is written exactly once.

# --- df_stage_lists (candidate pools) --------------------------------------
for col in ["desc_fr", "desc_en"]:
    df_stage_lists[col] = df_stage_lists[col].str.replace(SOURCE_TAG_RE, "", regex=True).str.strip()

# --- benchmark artifacts ---------------------------------------------------
pkl_vars(df_bench, "df_bench.pkl")
df_bench.to_csv(OUT_DIR / "benchmark.csv", index=False, encoding="utf-8")
print(f"Wrote {OUT_DIR / 'benchmark.csv'} ({len(df_bench)} rows)")

if unmatched:
    unmatched_path = OUT_DIR / "unmatched.csv"
    pd.DataFrame(unmatched).to_csv(unmatched_path, index=False, encoding="utf-8")
    print(f"Wrote {unmatched_path} ({len(unmatched)} rows)")

# --- stage-list artifacts --------------------------------------------------
stage_lists_path = OUT_DIR / "stage_lists.csv"
df_stage_lists.to_csv(stage_lists_path, index=False, encoding="utf-8")
print(f"Wrote {stage_lists_path} ({len(df_stage_lists)} rows)")

pkl_vars(df_stage_lists, "bbch_stage_lists.pkl")
pkl_vars(stage_lists, "bbch_stage_lists_dict.pkl")
pkl_vars(crop_candidates, "bbch_crop_candidates.pkl")
print(f"Cached stage lists + crop candidates to {VARS_DIR}")

Wrote /projects/agrifutur/sji/bsv-crossmodalIR/bbch-benchmark/benchmark/benchmark.csv (94 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/bbch-benchmark/benchmark/unmatched.csv (3 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/bbch-benchmark/benchmark/stage_lists.csv (161 rows)
Cached stage lists + crop candidates to /projects/agrifutur/sji/bsv-crossmodalIR/vars


## Step 6 — Verify

Runs **after** every artifact is written, so it validates the final data:

1. **Benchmark rows** — row count, rows per crop, empty-description counts, spot checks (citrus `59` → principal `5`; flax `D1_2` → TI code `D1`).
2. **Candidate pools** — sizes and empty-description counts per `(crop, level)`.
3. **Ground-truth coverage** — every benchmark row's stage id must exist in its own pool. Principal is asserted for BBCH crops only (citrus); TI rows carry a derived BBCH principal id that is informational and never evaluated.

In [8]:
# --- verification ----------------------------------------------------------
# Runs after every artifact is written, so it validates the final data.
# 1) benchmark rows: counts per crop, empty descriptions, spot checks
# 2) candidate pools: sizes and empty-description counts per (crop, level)
# 3) every benchmark ground-truth id must exist in its own candidate pool

print("=== Benchmark rows ===")
print(f"Rows: {len(df_bench)}   Unmatched: {len(unmatched)}")
print("\nRows per crop:")
print(df_bench["crop_name"].value_counts().to_string())

print("\nRows with empty secondary description (fr):",
      int((df_bench["secondary_desc_fr"] == "").sum()))
print("Rows with empty principal description (fr):",
      int((df_bench["principal_desc_fr"] == "").sum()))

print("\nSpot checks:")
spot = df_bench[df_bench["photo_name"].str.contains("bbch_59_1|LeBihan_D1_2", regex=True)]
display(spot)

print("\n=== Candidate pools ===")
print("Pool sizes per (crop, level):")
pool_sizes = (
    df_stage_lists.groupby(["crop_name", "level"]).size()
    .rename("n_candidates").reset_index()
)
print(pool_sizes.to_string(index=False))

print("\nEmpty descriptions per (crop, level):")
empty_desc = (
    df_stage_lists.assign(
        empty_fr=df_stage_lists["desc_fr"].str.strip() == "",
        empty_en=df_stage_lists["desc_en"].str.strip() == "",
    )
    .groupby(["crop_name", "level"])[["empty_fr", "empty_en"]].sum()
    .reset_index()
)
print(empty_desc.to_string(index=False))

# --- ground-truth coverage -------------------------------------------------
# Levels evaluated in notebook 32: principal for BBCH crops only (citrus), secondary for
# every crop. TI principal ids are derived BBCH digits that do not exist in the
# Smith-Froment letter scale, so they are reported but never asserted.
EVALUATED_LEVELS = {"bbch": ("principal", "secondary"), "ti": ("secondary",)}


def gt_stage_id(row, kind: str, level: str) -> str:
    """Ground-truth stage id of one benchmark row, in the pool's own vocabulary.

    - BBCH crops: BBCH codes (principal = 1 digit, secondary = 2 digits).
    - TI crops: the Terres Inovia code (`D1`, `E5`, ...) for the secondary level.
      The derived BBCH ids on TI rows are informational only and never evaluated.
    """
    if kind == "ti":
        return str(row["ti_stage_id"] or "")
    if level == "principal":
        return str(row["bbch_principal_stage_id"] or "")
    return str(row["bbch_secondary_stage_id"] or "")


coverage_rows, missing_rows = [], []
for _, row in df_bench.iterrows():
    crop = row["crop_name"]
    kind = CROP_CONFIG[crop]["kind"]
    for level in EVALUATED_LEVELS[kind]:
        pool_ids = {r["stage_id"] for r in stage_lists.get((crop, level), [])}
        gid = gt_stage_id(row, kind, level)
        in_pool = gid in pool_ids
        coverage_rows.append({"crop_name": crop, "level": level, "in_pool": in_pool})
        if not in_pool:
            missing_rows.append({"photo_name": row["photo_name"], "crop_name": crop,
                                 "level": level, "stage_id": gid})

coverage = pd.DataFrame(coverage_rows)
print("\nGround-truth coverage (evaluated levels):")
print(coverage.groupby(["crop_name", "level"])["in_pool"]
      .agg(n_in_pool="sum", n_rows="count").to_string())

if missing_rows:
    print("\nMISSING ground-truth ids:")
    print(pd.DataFrame(missing_rows).to_string(index=False))

assert not missing_rows, (
    f"{len(missing_rows)} benchmark rows have a ground-truth id absent from their pool"
)

# Informational: TI rows carry a derived BBCH principal id, which is not evaluated.
n_ti = int((df_bench["crop_name"].map(lambda c: CROP_CONFIG[c]["kind"]) == "ti").sum())
if n_ti:
    print(f"\nNote: {n_ti} TI rows carry a derived BBCH principal id "
          f"(principal level is not evaluated for TI crops).")

print("\nAll checks passed.")

=== Benchmark rows ===
Rows: 94   Unmatched: 3

Rows per crop:
crop_name
citrus         81
oilseedFlax    11
oilseedRape     2

Rows with empty secondary description (fr): 0
Rows with empty principal description (fr): 2

Spot checks:


,photo_name,crop_name,bbch_principal_stage_id,bbch_secondary_stage_id,ti_stage_id,principal_desc_fr,principal_desc_en,secondary_desc_fr,secondary_desc_en
32,citrus#photo_citrus_bbch_59_1.jpg,citrus,5,59,,stade principal BBCH 5 des agrumes. Développem...,principal stage BBCH 5 of citrus. Inflorescenc...,stade secondaire BBCH 59 des agrumes. La plupa...,secondary stage BBCH 59 of citrus. Most flower...
87,oilseedFlax#terres_inovia_Zoe_LeBihan_D1_2_pho...,oilseedFlax,3,31,D1,stade principal BBCH 3 du lin. Elongation de l...,principal stage BBCH 3 of linen. Stem extension,stade secondaire Terres Inovia D1 du lin. Haut...,Terres Inovia secondary stage D1 of oilseed fl...



=== Candidate pools ===
Pool sizes per (crop, level):
  crop_name     level  n_candidates
     citrus principal             8
     citrus secondary            35
oilseedFlax principal             9
oilseedFlax secondary            81
oilseedRape secondary            28

Empty descriptions per (crop, level):
  crop_name     level  empty_fr  empty_en
     citrus principal         0         0
     citrus secondary         0         0
oilseedFlax principal         0         0
oilseedFlax secondary         0         0
oilseedRape secondary         0         0

Ground-truth coverage (evaluated levels):
                       n_in_pool  n_rows
crop_name   level                       
citrus      principal         81      81
            secondary         81      81
oilseedFlax secondary         11      11
oilseedRape secondary          2       2

Note: 13 TI rows carry a derived BBCH principal id (principal level is not evaluated for TI crops).

All checks passed.
